# Evaluate by-instance WSD dataset with Dotted-WSD

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
!pip install CwnGraph

In [ ]:
import json
import sys

import numpy as np
import pandas as pd
import torch
import wandb
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import AutoTokenizer

sys.path.append("./dotted-wsd/GlossBERT")
from data_collator import DataCollatorForDottedWSD
from deberta_dotted_wsd import DottedWSD  # 改成新的script
from eval_dotted_wsd import evaluate_dotted_wsd

wandb.login()

## Data Hash

```
./data/WSD_merge_test_v2.csv  : e1e519
```

In [ ]:
########################### 儲存 pretrained model的資料夾 ###########################
DATE = "241019"
ID = "2000"
PRETRAINED_MODEL_PATH = f"/mnt/md0/cckk2913/model/{DATE}-{ID}/ep2_24095"
########################################################################

config = {
    "batch_size": 32,
    "notebook": "20.44",
}

model_id = "MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7"
tokenizer = AutoTokenizer.from_pretrained(model_id)
BATCH_SIZE = config["batch_size"]

In [ ]:
# Read test data
wsdtest = "./dotted-wsd/GlossBERT/data/WSD_merge_test_v2.csv"

CHECK_DATA_HASH = True
if CHECK_DATA_HASH:
    import hashlib
    from pathlib import Path

    for data_path in (wsdtest,):
        hasher = hashlib.sha1()
        hasher.update(Path(data_path).read_bytes())
        h = hasher.digest().hex()[:6]
        print(f"{data_path:<30s}: {h}")

In [ ]:
from CwnGraph import CwnBase


class WSDDataset(Dataset):
    def __init__(self, datapath):
        self.data = pd.read_csv(
            datapath, dtype={"test_sense_id": str, "cwn_sense_id": str, "test_definition": str}
        )
        self.cwn = CwnBase()
        self.instances = self.preprocess(self.data)

    def __len__(self):
        return len(self.instances)

    def __getitem__(self, idx):
        return {k: self.instances[idx][k] for k in ("candidate", "context", "label", "example_id")}

    def preprocess(self, data):
        # [CLS] <instance> [SEP] <word>,<candidate_sense>,<candidate_sense例句>
        # [CLS] s['sentence_id'] [SEP] s['target_word_id'] [COMMA] ['cwn_definition_id'] [COMMA] s['cwn_sentence_id'] [SEP]

        # drop those examples not having a correct answer (happens ~0.1% in annotation data)
        exid_sum = data.groupby("example_id").apply(lambda x: x.label.sum())
        excl_ids = exid_sum[exid_sum == 0].index.values
        data = data.loc[~data.example_id.isin(excl_ids)]

        instances = []
        id_map = {}
        for _, row in tqdm(data.iterrows(), total=data.shape[0]):
            word = row["test_word"]
            sentence = row["test_sentence"]
            candidate_sense = row["cwn_definition"]
            sense_def = row["test_definition"]
            cand_ex = row["cwn_sentence"]
            example_id = row["example_id"]
            context = sentence
            candidate = f"{word},{candidate_sense},{cand_ex}"
            label = row["label"]
            if not isinstance(row["test_sense_id"], str):
                label_key = f"{word}:{sense_def}"
                if label_key not in id_map:
                    senses = self.cwn.find_all_senses(word)
                    test_sense_id = [x.id for x in senses if x.definition == sense_def][0]
                    id_map[label_key] = test_sense_id
                test_sense_id = id_map[label_key]
            else:
                test_sense_id = row["test_sense_id"]
            # example_id is offset by 10000 to leave room for RP dataset
            instance: WSDInstance = {
                "word": word,
                "sense_id": test_sense_id,
                "pos": row["test_pos"],
                "context": context,
                "candidate": candidate,
                "example_id": 10000 + example_id,
                "data_source": "wsd",
                "label": int(label),
            }

            instances.append(instance)

        return instances

In [ ]:
# Data loader
wsdtestset = WSDDataset(wsdtest)
collate_fn = DataCollatorForDottedWSD(tokenizer)
wsd_evalloader = DataLoader(wsdtestset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

In [ ]:
wandb.init(project="dotted-wsd", config=config)

In [ ]:
def evaluate(model, dataloader):
    model.eval()
    all_logits = []
    all_labels = []
    all_exids = []
    eval_loss_vec = []

    for batch in tqdm(dataloader, desc="Eval"):
        batch.to(device)
        with torch.no_grad():
            out = model(**batch)
            eval_loss_vec.append(out.loss.detach().item())
        logits = out.logits.detach().cpu().tolist()
        labels = batch["labels"].cpu().tolist()
        example_ids = batch["example_ids"].cpu().tolist()
        all_logits.extend(logits)
        all_labels.extend(labels)
        all_exids.extend(example_ids)
    eval_out = evaluate_dotted_wsd(np.array(all_logits), np.array(all_exids), np.array(all_labels))
    eval_loss = sum(eval_loss_vec) / len(eval_loss_vec)

    print(f"Evaluation acc: {eval_out[0]:.4f}")
    return eval_out, all_logits

## Evaluation

In [ ]:
model = DottedWSD(model_id)
model.load_state_dict(torch.load(PRETRAINED_MODEL_PATH, weights_only=True))
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
print(f"Device: {device}")

In [ ]:
wsd_eval, wsd_logits = evaluate(model, wsd_evalloader)

In [ ]:
wandb.finish()

In [ ]:
assert len(wsd_logits) == len(wsdtestset)

In [ ]:
for inst_x, logit_x in zip(wsdtestset.instances, wsd_logits):
    inst_x["logit"] = logit_x

In [ ]:
wsd_test_eval = pd.DataFrame.from_dict(wsdtestset.instances)

In [ ]:
wsd_test_eval.head()

In [ ]:
wsd_test_eval.to_csv("./data/deberta_wsd-eval-by-instance.csv", encoding="utf-8", index=False)

## Calculate metrics

### 1. 

concat OLD wsd_instances_evals.csv & deberta_wsd-eval-by-instance.csv

extracting `word,sense_id,pos,example_id,n_candidate,`

In [ ]:
old_wsd = pd.read_csv("./dotted-wsd/data/wsd_instances_evals.csv")
old_wsd = old_wsd.drop(["label", "pred"], axis=1)
old_wsd.head()

In [ ]:
wsd_test_eval = pd.read_csv("./data/deberta_wsd-eval-by-instance.csv")
wsd_test_eval.head()

In [ ]:
merge_wsd_test_eval = pd.merge(old_wsd, wsd_test_eval)
merge_wsd_test_eval.head()

### 2.

compute example eval

In [ ]:
def compute_example(group):
    n_candidate = group.shape[0]
    label_id = np.where(group.label == 1)[0][0]
    pred_id = np.argmax(group.logit)
    row0 = group.iloc[0]
    return pd.Series(
        {
            "word": row0.word,
            "sense_id": row0.sense_id,
            "pos": row0.pos,
            "example_id": row0.example_id,
            "n_candidate": n_candidate,
            "label": label_id,
            "pred": pred_id,
        }
    )


example_evals = merge_wsd_test_eval.groupby("example_id").apply(compute_example)

In [ ]:
example_evals.head()

In [ ]:
wsd_inevals_acc = (example_evals.label == example_evals.pred).sum() / example_evals.shape[0]
wsd_inevals_acc

In [ ]:
(
    example_evals.assign(acc=example_evals.label == example_evals.pred)
    .loc[:, ["n_candidate", "acc"]]
    .groupby("n_candidate")
    .mean("acc")
    .reset_index()
).plot.scatter("n_candidate", "acc")

## Output

In [ ]:
eval_path = "./data/deberta_20.44_wsd_instances_evals.csv"
example_evals.to_csv(eval_path, index=False)

In [ ]:
example_evals.shape

In [ ]:
metric_path = "./data/metrics/deberta_wsd_instances_acc.json"
with open(metric_path, "w") as fout:
    json.dump({"wsd_inevals_acc": wsd_inevals_acc}, fout)

In [ ]:
import hashlib

for path_x in (eval_path, metric_path):
    h = hashlib.sha1()
    h.update(Path(path_x).read_bytes())
    print(path_x, h.hexdigest()[:6])

## Divide by Word class

In [ ]:
import pandas as pd

df = pd.read_csv("./data/deberta_20.44_wsd_instances_evals.csv")
df.head()

In [ ]:
# Step 2: Classify 'pos' as 'V', 'N', or 'others', treating lowercase 'n' as 'N'
df["group"] = df["pos"].apply(
    lambda x: "V" if x.startswith("V") else "N" if x.startswith(("N", "n")) else "others"
)

# Step 3: Check if 'label' matches 'pred' (i.e., calculate correctness)
df["correct"] = df["label"] == df["pred"]

# Step 4: Calculate the accuracy for each group
grouped_accuracy = df.groupby("group")["correct"].mean()

# Output the results
print(df)  # The dataframe with 'group' and 'correct' columns
print(grouped_accuracy)  # The accuracy for each group